# 02 · Conexión con Google Earth Engine (proyecto `viirs-peru`)

**Objetivo de este notebook (solo conexión):**
1. Inicializar GEE con el proyecto `viirs-peru`.
2. Usar la geometría de la **provincia de Cusco** como área de estudio.
3. Inventariar, por año, las escenas de **Sentinel-2** y **Landsat** y su cobertura de nubes.
4. Verificar que **ESA WorldCover** y **SRTM (DEM)** cargan sobre el área.

**Notas de lectura:**
- Una *escena* es una imagen de un satélite en una fecha y un tile. El conteo puede incluir solapes entre órbitas.
- La *nube* es el porcentaje de nubes **de toda la escena** (metadato). No es la nube dentro de la provincia. Para eso hará falta enmascaramiento en el paso de procesamiento.
- 2026 es un año parcial (hasta la fecha de ejecución).

**Requisitos:** `earthengine-api`, `geemap`, `geopandas`, `pandas`, `matplotlib`. La primera vez se abrirá el navegador para autorizar la cuenta de Google.

## Mapa de fuentes del proyecto

| # | Fuente | Estado | Sección | Qué falta |
|---|---|---|---|---|
| 1 | Sentinel-2 (imágenes, vía GEE) | Inventario conectado | 4 y 5 | Decidir si el procesamiento usa la copia de GEE o Copernicus Data Space directo |
| 2 | Landsat 5/7/8/9 (vía GEE) | Inventario conectado | 4 y 5 | Nada para el inventario. Falta enmascaramiento de nubes para el procesamiento |
| 3 | ESA WorldCover 2020 y 2021 | Conectada | 8 | Nada |
| 4 | SRTM (modelo de elevación) | Conectada | 9 | Nada |
| 5 | MapBiomas Perú, Colección 4 (referencia, vía GEE) | Conectada | 10 | Nada. Solo verificar la ruta del asset y el año elegido |
| 6 | GEOBOSQUES, pérdida de bosque (MINAM) | **Pendiente** | 11 | Descargar el archivo y guardarlo en `data/raw/` |
| 7 | Red vial | **Pendiente** | 12 | Elegir la fuente (por ejemplo OpenStreetMap o MTC) |
| 8 | Límites administrativos | Parcial | 13 | Provincia lista en el notebook 01. Falta el archivo de distritos |

Las secciones con celda vacía son las pendientes. Se completan cuando se confirme la fuente.

## 1. Configuración

In [ ]:
from pathlib import Path

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

GEE_PROJECT = "viirs-peru"          # ID del proyecto en Google Cloud
GEE_PROJECT_NUMBER = 815270851429   # número del proyecto (referencia)
ANIO_INI, ANIO_FIN = 2011, 2026     # 15 años + 2026 parcial


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la que contiene data/raw."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw").is_dir():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw desde esta ubicación.")


print("Proyecto GEE:", GEE_PROJECT, "| número:", GEE_PROJECT_NUMBER)

## 2. Inicializar Earth Engine

Si es la primera vez, `ee.Authenticate()` abre el navegador. Después basta con `ee.Initialize`.

In [ ]:
def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()  # primera vez: autorizar la cuenta en el navegador
        ee.Initialize(project=proyecto)


iniciar_gee()
print("GEE conectado.")
print("Prueba de cálculo en servidor (2 + 3):", ee.Number(2).add(3).getInfo())

## 3. Área de estudio: provincia de Cusco

Se lee el mismo `PROVINCIA.gpkg` de `data/raw/` (solo lectura) y se filtra por códigos (`ccdd=08`, `ccpp=01`).
La geometría se simplifica ~10 m para aligerar las peticiones a GEE.

In [ ]:
RAIZ = buscar_raiz()
GPKG = RAIZ / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg"

gdf = gpd.read_file(GPKG)
prov = gdf[(gdf["ccdd"] == "08") & (gdf["ccpp"] == "01")]
assert len(prov) == 1, "El filtro de provincia no devolvió exactamente 1 registro"

geom_prov = prov.geometry.iloc[0].simplify(0.0001, preserve_topology=True)
aoi = ee.Geometry(geom_prov.__geo_interface__)

print("Provincia:", prov["nombprov"].iloc[0], "-", prov["nombdep"].iloc[0])
print("Área del AOI (km²):", round(aoi.area().divide(1e6).getInfo(), 1))

## 4. Inventario anual de escenas y nubes

Se consulta cada año por separado para Sentinel-2 (`S2_SR_HARMONIZED`) y Landsat (C2 L2: Landsat 5, 7, 8 y 9).

In [ ]:
COLECCIONES = {
    "Sentinel-2": ("COPERNICUS/S2_SR_HARMONIZED", "CLOUDY_PIXEL_PERCENTAGE", "Sentinel-2"),
    "Landsat 5": ("LANDSAT/LT05/C02/T1_L2", "CLOUD_COVER", "Landsat"),
    "Landsat 7": ("LANDSAT/LE07/C02/T1_L2", "CLOUD_COVER", "Landsat"),
    "Landsat 8": ("LANDSAT/LC08/C02/T1_L2", "CLOUD_COVER", "Landsat"),
    "Landsat 9": ("LANDSAT/LC09/C02/T1_L2", "CLOUD_COVER", "Landsat"),
}


def inventario(nombre, coleccion_id, prop_nube, familia, anios):
    base = ee.ImageCollection(coleccion_id).filterBounds(aoi)
    filas = []
    for y in anios:
        col = base.filterDate(f"{y}-01-01", f"{y + 1}-01-01")
        r = ee.Dictionary({
            "n": col.size(),
            "nube": col.aggregate_mean(prop_nube),
        }).getInfo()
        filas.append({
            "año": y,
            "plataforma": nombre,
            "familia": familia,
            "escenas": r["n"],
            "nube_media_%": r["nube"],
        })
    return filas


anios = list(range(ANIO_INI, ANIO_FIN + 1))
filas = []
for nombre, (cid, prop, fam) in COLECCIONES.items():
    print("Consultando", nombre, "...")
    filas += inventario(nombre, cid, prop, fam, anios)

df = pd.DataFrame(filas)
print("Listo:", len(df), "filas")

## 5. Tabla resumen por año

Por familia (Sentinel-2 vs. Landsat). La nube media de Landsat está ponderada por número de escenas.

In [ ]:
df["nube_x_escenas"] = df["nube_media_%"].fillna(0) * df["escenas"]

resumen = (
    df.groupby(["año", "familia"], as_index=False)
      .agg(escenas=("escenas", "sum"), nube_x=("nube_x_escenas", "sum"))
)
resumen["nube_media_%"] = resumen["nube_x"] / resumen["escenas"].where(resumen["escenas"] > 0)

tabla_escenas = resumen.pivot(index="año", columns="familia", values="escenas").fillna(0).astype(int)
tabla_nube = resumen.pivot(index="año", columns="familia", values="nube_media_%").round(1)
tabla = tabla_escenas.join(tabla_nube, lsuffix=" escenas", rsuffix=" nube %")

tabla

## 6. Detalle por plataforma (escenas)

In [ ]:
df.pivot(index="año", columns="plataforma", values="escenas").fillna(0).astype(int)

## 7. Gráfico por año

Izquierda: escenas por año. Derecha: nube media por año. Dos series con leyenda; los colores están validados para distinguirse en visión normal y con deuteranopia/protanopia.

In [ ]:
colores = {"Sentinel-2": "#1a6fc0", "Landsat": "#d9731a"}

fig, ejes = plt.subplots(1, 2, figsize=(15, 5))

x = np.arange(len(tabla_escenas.index))
ancho = 0.4
for i, (fam, color) in enumerate(colores.items()):
    ejes[0].bar(x + (i - 0.5) * ancho, tabla_escenas[fam], ancho, color=color, label=fam)
ejes[0].set_xticks(x, tabla_escenas.index, rotation=90)
ejes[0].set_title("Escenas por año")
ejes[0].set_ylabel("Nº de escenas")
ejes[0].legend(frameon=False)

for fam, color in colores.items():
    ejes[1].plot(tabla_nube.index, tabla_nube[fam], marker="o", color=color, label=fam)
ejes[1].set_title("Cobertura de nubes media por año (escena completa)")
ejes[1].set_ylabel("% de nubes")
ejes[1].set_ylim(0, 100)
ejes[1].legend(frameon=False)

for ax in ejes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e5e5e5", linewidth=0.6)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 8. ESA WorldCover (referencia de cobertura)

Se verifica la carga de WorldCover 2020 (v100) y 2021 (v200) sobre el AOI. Se usa el mosaico de los tiles que cruzan la provincia, recortado al AOI, y se cuenta el área por clase a 100 m (solo para comprobar que carga).

In [ ]:
NOMBRES_WC = {
    10: "Árboles", 20: "Arbustos", 30: "Pastizales", 40: "Cultivos",
    50: "Urbano", 60: "Suelo desnudo", 70: "Nieve/hielo", 80: "Agua",
    90: "Humedales herbáceos", 95: "Manglares", 100: "Musgo/liquen",
}


def histograma_worldcover(coleccion_id):
    img = ee.ImageCollection(coleccion_id).filterBounds(aoi).mosaic().select("Map")
    hist = img.clip(aoi).reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=aoi,
        scale=100,
        maxPixels=1e10,
        bestEffort=True,
    ).get("Map").getInfo()
    if not hist:
        raise ValueError(f"WorldCover no devolvió datos en {coleccion_id}")
    return hist


filas_wc = []
for etiqueta, cid in [("WorldCover 2020 (v100)", "ESA/WorldCover/v100"),
                      ("WorldCover 2021 (v200)", "ESA/WorldCover/v200")]:
    hist = histograma_worldcover(cid)
    total = sum(hist.values())
    print(f"✔ {etiqueta} cargó · {int(total)} píxeles de 100 m en el AOI")
    for k, v in hist.items():
        filas_wc.append({"producto": etiqueta,
                         "clase": NOMBRES_WC.get(int(float(k)), k),
                         "%": round(100 * v / total, 1)})

pd.DataFrame(filas_wc).pivot(index="clase", columns="producto", values="%").fillna(0)

## 9. SRTM (modelo de elevación)

Se verifica la carga de `USGS/SRTMGL1_003` (30 m, referencia de elevación) y se muestran altitudes mínima, media y máxima de la provincia.

In [ ]:
dem = ee.Image("USGS/SRTMGL1_003")

stats_dem = dem.clip(aoi).reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=aoi,
    scale=90,
    maxPixels=1e10,
    bestEffort=True,
).getInfo()

print("✔ SRTM cargó sobre el AOI")
print(f"  Altitud mínima: {stats_dem['elevation_min']:.0f} m")
print(f"  Altitud media:  {stats_dem['elevation_mean']:.0f} m")
print(f"  Altitud máxima: {stats_dem['elevation_max']:.0f} m")

## 10. MapBiomas Perú: cobertura y uso del suelo (Colección 4)

Asset público en GEE: `projects/mapbiomas-public/assets/peru/collection4/mapbiomas_peru_collection4_coverage_v1`.
Tiene una banda por año (`classification_YYYY`), de 1985 a 2025, a 30 m. Se usa como referencia para comparar con nuestros resultados.

Cambia `ANIO_MB` en la celda 10.1 para ver otro año.

### 10.1 Cargar el asset y revisar las bandas

In [ ]:
MB_ASSET = "projects/mapbiomas-public/assets/peru/collection4/mapbiomas_peru_collection4_coverage_v1"
ANIO_MB = 2025  # año a visualizar (1985 a 2025)

mb = ee.Image(MB_ASSET)
bandas_mb = mb.bandNames().getInfo()
print("Bandas disponibles:", len(bandas_mb))
print(bandas_mb)

BANDA_MB = f"classification_{ANIO_MB}"
assert BANDA_MB in bandas_mb, f"No encuentro la banda {BANDA_MB}. Revisa la lista de bandas de arriba."
clasif_mb = mb.select(BANDA_MB)
print("Banda seleccionada:", BANDA_MB)

### 10.2 Leyenda de la Colección 4

Códigos y colores de la leyenda corta oficial de MapBiomas Perú Colección 4 (solo clases hoja, que son las que aparecen como valores del raster).

In [ ]:
LEYENDA_MB = {
    3: ("Bosque", "#1f8d49"),
    4: ("Bosque seco", "#7dc975"),
    5: ("Manglar", "#04381d"),
    6: ("Bosque inundable", "#007785"),
    51: ("Herbazal inundable de tierras bajas", "#5faf92"),
    82: ("Herbazal inundable altoandino", "#26abab"),
    12: ("Formación herbácea", "#d6bc74"),
    66: ("Matorral y otros arbustales", "#a89358"),
    70: ("Loma costera (beta)", "#be9e00"),
    21: ("Mosaico agropecuario", "#ffefc3"),
    9: ("Plantación forestal", "#7a6c00"),
    35: ("Palma aceitera", "#9065d0"),
    40: ("Arroz (beta)", "#c71585"),
    15: ("Pasto (beta)", "#edde8e"),
    24: ("Infraestructura urbana", "#d4271e"),
    30: ("Minería", "#9c0027"),
    23: ("Playa", "#ffa07a"),
    92: ("Superficie rocosa", "#d98a45"),
    52: ("Superficie salina costera", "#f0b4a8"),
    61: ("Superficie de salar", "#f5d5d5"),
    68: ("Otra área natural sin vegetación", "#e97a7a"),
    25: ("Otra área antrópica sin vegetación", "#db4d4f"),
    33: ("Río, lago u océano", "#2532e4"),
    31: ("Acuicultura", "#091077"),
    34: ("Glaciar", "#93dfe6"),
    27: ("No observado", "#ffffff"),
}
print("Clases en la leyenda:", len(LEYENDA_MB))

### 10.3 Área por clase dentro de la provincia de Cusco

In [ ]:
clasif_mb_aoi = clasif_mb.clip(aoi)

# Área (m²) por código de clase, dentro del AOI
grupos_mb = (
    ee.Image.pixelArea().addBands(clasif_mb_aoi)
    .reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName="codigo"),
        geometry=aoi,
        scale=30,
        maxPixels=1e10,
        bestEffort=True,
    )
    .get("groups")
    .getInfo()
)

filas_mb = []
for g in grupos_mb:
    codigo = int(g["codigo"])
    nombre, _ = LEYENDA_MB.get(codigo, (f"Código {codigo} (no está en la leyenda)", "#999999"))
    filas_mb.append({"código": codigo, "clase": nombre, "área_km2": g["sum"] / 1e6})

tabla_mb = pd.DataFrame(filas_mb).sort_values("área_km2", ascending=False).reset_index(drop=True)
tabla_mb["%"] = (100 * tabla_mb["área_km2"] / tabla_mb["área_km2"].sum()).round(1)
tabla_mb["área_km2"] = tabla_mb["área_km2"].round(2)
print(f"Año {ANIO_MB} · área total: {tabla_mb['área_km2'].sum():.1f} km²")
tabla_mb

### 10.4 Mapa interactivo

Se muestra la clasificación del año elegido sobre la provincia de Cusco, con su límite en negro y la leyenda de las clases presentes.

In [ ]:
import geemap

# Paleta indexada por código (0 a 92), con blanco en los valores no usados
pal_mb = ["ffffff"] * 93
for codigo, (_, color) in LEYENDA_MB.items():
    pal_mb[codigo] = color.lstrip("#")
vis_mb = {"min": 0, "max": 92, "palette": pal_mb}

mapa_mb = geemap.Map()
mapa_mb.centerObject(aoi, 9)
mapa_mb.addLayer(clasif_mb_aoi, vis_mb, f"MapBiomas Perú C4 {ANIO_MB}")
mapa_mb.addLayer(
    ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(aoi)]), 1, 2),
    {"palette": "000000"},
    "Límite provincia Cusco",
)

presentes = tabla_mb["código"].tolist()
leyenda_mb = {f"{LEYENDA_MB[c][0]} ({c})": LEYENDA_MB[c][1] for c in presentes if c in LEYENDA_MB}
mapa_mb.add_legend(title=f"MapBiomas Perú C4 · {ANIO_MB}", legend_dict=leyenda_mb)

mapa_mb

### 10.5 Exportar el GeoTIFF (opcional)

Por defecto está desactivado. Al activarlo, GEE genera el GeoTIFF y lo envía a tu Google Drive, carpeta `GEE_exports`. Después descárgalo y guárdalo en `data/raw/MapBiomas/`. Revisa el progreso en la pestaña **Tasks** de code.earthengine.google.com.

In [ ]:
EXPORTAR_MB = False  # cambia a True para enviar la exportación

if EXPORTAR_MB:
    tarea_mb = ee.batch.Export.image.toDrive(
        image=clasif_mb_aoi,
        description=f"mapbiomas_peru_c4_{ANIO_MB}_cusco",
        folder="GEE_exports",
        region=aoi,
        scale=30,
        maxPixels=1e10,
        fileFormat="GeoTIFF",
    )
    tarea_mb.start()
    print("Exportación enviada a Google Drive (carpeta GEE_exports).")
else:
    print("Exportación desactivada. Pon EXPORTAR_MB = True para generar el GeoTIFF.")

## 11. GEOBOSQUES: pérdida de bosque (MINAM)

**Pendiente.** Los datos se descargan del portal del MINAM. Guardar el archivo en `data/raw/` antes de leerlo aquí.

## 12. Red vial (auxiliar)

**Pendiente.** Elegir la fuente de la red vial y su formato.

## 13. Límites administrativos: distritos (auxiliar)

**Pendiente.** La provincia ya está en el notebook 01. Falta el archivo de distritos para la provincia de Cusco.

## Resumen

- Conexión a GEE con `viirs-peru` y AOI = provincia de Cusco (`ccdd=08`, `ccpp=01`).
- Inventario anual de Sentinel-2 y Landsat (2011–2026) con nube media por escena.
- ESA WorldCover 2020/2021 y SRTM cargan sobre el área.
- MapBiomas Perú Colección 4 (1985 a 2025) cargado desde GEE: bandas, área por clase, mapa interactivo y exportación opcional.
- Pendientes: GEOBOSQUES, red vial y distritos (secciones 11 a 13).

Con el inventario ya puedes decidir el periodo definitivo.